# Preprocessing and Label Creation
This notebook will load the datasets, clean them, apply labels based on the true malicious windows, and save the processed files.

In [1]:
import pandas as pd
import os
import sys

# Ensure src module is discoverable
sys.path.append(os.path.abspath('..'))

from src.load_data import load_logon, load_file_activity, load_device
from src.preprocess import clean_dataframe, load_insider_labels, apply_labels, save_processed_data

In [2]:
# Load datasets
logon_df = load_logon('../data')
file_df = load_file_activity('../data')
device_df = load_device('../data')

In [3]:
# Load insider labels
insiders_df = load_insider_labels('../data/answers/insiders.csv')
print("Insiders Ground Truth Table:")
display(insiders_df.head())

Insiders Ground Truth Table:


,user,start,end
0,ONS0995,2010-03-06 01:41:56,2010-03-20 08:10:12
1,CSF0929,2010-07-01 01:24:58,2010-07-16 06:52:00
2,CCH0959,2010-08-02 10:34:31,2010-09-30 15:04:03
3,RCW0822,2010-09-29 21:10:27,2010-10-15 06:34:52
4,JCE0258,2010-07-12 08:16:02,2010-09-03 16:16:29


In [4]:
def process_and_evaluate(df, name):
    print(f"--- Processing {name} ---")
    print(f"Original shape: {df.shape}")
    
    df_clean = clean_dataframe(df)
    print(f"Cleaned shape: {df_clean.shape}")
    
    df_labeled = apply_labels(df_clean, insiders_df)
    
    print("Label counts:")
    print(df_labeled['label'].value_counts())
    
    malicious_activity = df_labeled[df_labeled['label'] == 1]
    print(f"Found {len(malicious_activity)} malicious events.")
    if len(malicious_activity) > 0:
        display(malicious_activity.head())
        
    save_processed_data(df_labeled, f"{name}_processed.csv", '../data/processed')
    return df_labeled

In [5]:
# Apply pipeline
logon_processed = process_and_evaluate(logon_df, 'logon')
file_processed = process_and_evaluate(file_df, 'file_activity')
device_processed = process_and_evaluate(device_df, 'device')

--- Processing logon ---
Original shape: (854859, 5)
Cleaned shape: (854859, 5)


Label counts:
label
0    851411
1      3448
Name: count, dtype: int64
Found 3448 malicious events.


,id,date,user,pc,activity,label
294803,{T2N6-X9QL06RX-0762MGAR},2010-06-10 13:13:31,CSC0217,PC-5391,Logon,1
294826,{K7J1-R9NA31CC-0202USEF},2010-06-10 13:28:49,CSC0217,PC-5391,Logoff,1
294906,{Q9F3-J3GS01PY-1396BSPW},2010-06-10 15:33:00,CSC0217,PC-6377,Logoff,1
295379,{O6P2-Y9ZZ98NN-2331PRFQ},2010-06-10 17:21:26,CSC0217,PC-9999,Logon,1
295380,{F2G7-P0FF01QA-5034JQWM},2010-06-10 17:21:56,CSC0217,PC-6377,Logon,1


Saved logon_processed.csv to ../data/processed/
--- Processing file_activity ---
Original shape: (445581, 6)
Cleaned shape: (445581, 6)


Label counts:
label
0    441934
1      3647
Name: count, dtype: int64
Found 3647 malicious events.


,id,date,user,pc,filename,content,label
153762,{E5M5-D6KI89WL-2909ZPOD},2010-06-10 15:20:36,CSC0217,PC-6377,6UQIYOYG.exe,4D-5A-90-00-03-00-00-00-04-00-00-00-FF-FF-00-0...,1
156542,{R6S2-E6SR42RP-8679BIPE},2010-06-14 15:34:07,PNL0301,PC-5611,NXAT771C.txt,36-30-57-49 unsuccessfully march directly litt...,1
156558,{G6N4-S5RB20IP-1927TMNV},2010-06-14 15:41:26,PNL0301,PC-5611,HW1ZDMHN.pdf,25-50-44-46-2D research common anecdotes ara i...,1
156566,{H5G9-S1KM75IC-0767JFJW},2010-06-14 15:45:16,PNL0301,PC-5611,8SD2EH7T.pdf,25-50-44-46-2D maine scene related rise many 8...,1
156572,{Y7B1-V7RX37TL-6683JNEW},2010-06-14 15:47:53,PNL0301,PC-5611,DA7NY8LS.pdf,25-50-44-46-2D pursues considered just world c...,1


Saved file_activity_processed.csv to ../data/processed/
--- Processing device ---
Original shape: (405380, 5)
Cleaned shape: (405380, 5)


Label counts:
label
0    398293
1      7087
Name: count, dtype: int64
Found 7087 malicious events.


,id,date,user,pc,activity,label
140547,{Q1U6-Z0VP50BW-1523RIES},2010-06-10 15:18:19,CSC0217,PC-6377,Connect,1
140557,{R9C9-D7OJ27KI-7697NZPR},2010-06-10 15:22:06,CSC0217,PC-6377,Disconnect,1
140784,{A9P6-K4UQ50BO-3732TDXK},2010-06-10 18:35:02,CSC0217,PC-5866,Connect,1
140786,{A7E7-P5MQ60QN-4135CTGS},2010-06-10 18:38:04,CSC0217,PC-5866,Disconnect,1
143215,{C8D7-E8HB76ZF-5246NKXF},2010-06-14 15:25:39,PNL0301,PC-5611,Connect,1


Saved device_processed.csv to ../data/processed/
